# Схлопывание целых тар
Запустите все ячейки сверху вниз. Исходная таблица не изменяется.

Адрес `B3-01.02.03`: B3 — этаж/зона, 01 — улица, 02 — стеллаж, 03 — полка.
Короткий формат `Б1.02.03` также поддерживается (улица 02).

Приоритет пар: **код → группа в одном секторе → сектор**. Без общего сектора переноса нет.
При равном приоритете сначала выбирается ближайшая улица, затем более полная итоговая тара.
Доля заполнения = сумма количества / вместимости каждого товара; максимум 100%.
Это оценка объёма по средней вместимости, а не гарантия физической укладки.

Переносятся целые тары с количеством до `MAX_SMALL_QTY`; `None` снимает ограничение.
Получатель больше не переносится: цепочек и циклов нет. Алгоритм жадный и не гарантирует
минимально возможное количество оставшихся тар. Приоритет определяется по общим товарам
в составе тар; смешанные тары не разбиваются.

Лист «Вместимость секторов» показывает средние значения исходного столбца «Вместимость» (равный вес каждого кода). Алгоритм сохраняет вместимость конкретной строки; если внутри сектора она различается, среднее справочное значение её не заменяет.


In [ ]:
from pathlib import Path
import re
import pandas as pd


def parse_address(value, street_component='after_dash'):
    """Возвращает (этаж/зона, улица); неизвестные форматы запрещены."""
    text = str(value).strip().upper()
    extended = re.fullmatch(r'([^\s.]+)-(\d+)\.(\d+)\.(\d+)', text)
    simple = re.fullmatch(r'([^\s.]+)\.(\d+)\.(\d+)', text)
    if extended:
        zone, first, second, shelf = extended.groups()
        if street_component == 'after_dash':
            return zone, int(first)
        if street_component == 'middle':
            return f'{zone}-{first}', int(second)
        raise ValueError('street_component: after_dash или middle')
    if simple:
        zone, street, shelf = simple.groups()
        return zone, int(street)
    raise ValueError(f'Неизвестный формат адреса: {value!r}')


def optimize_whole_bins(df, max_small_qty=6, max_street_distance=1,
                        street_component='after_dash'):
    """Перенос целых тар. Приоритет: код > группа в секторе > сектор.

    Заполнение = сумма(Количество / Вместимость). Вместимость — число
    единиц данного товара, занимающих целую тару. Дробить тары нельзя.
    max_small_qty=None разрешает рассматривать все тары как источники.
    Неподвижные получатели исключают цепочки и циклы переносов.
    """
    required = ['Тара', 'Код', 'Сектор', 'Группа', 'Количество', 'Вместимость', 'Адрес']
    missing = set(required) - set(df.columns)
    if missing:
        raise ValueError(f'Отсутствуют столбцы: {sorted(missing)}')
    if max_street_distance < 0 or int(max_street_distance) != max_street_distance:
        raise ValueError('Расстояние улиц должно быть целым неотрицательным числом')
    if max_small_qty is not None and max_small_qty <= 0:
        raise ValueError('Порог количества должен быть положительным или None')
    result = df.copy().reset_index(drop=True)
    if result[required].isna().any().any():
        raise ValueError('Обязательные столбцы содержат пропуски')
    for col in ['Количество', 'Вместимость']:
        result[col] = pd.to_numeric(result[col], errors='raise')
        if not result[col].map(lambda v: float('-inf') < v < float('inf')).all():
            raise ValueError(f'{col}: обнаружены бесконечные значения')
    if (result['Количество'] < 0).any() or (result['Вместимость'] <= 0).any():
        raise ValueError('Количество должно быть >= 0, вместимость > 0')
    normalize = lambda v: str(v).strip().casefold()
    bins = {}
    for bin_id, rows in result.groupby('Тара', sort=False):
        locations = {parse_address(v, street_component) for v in rows['Адрес']}
        if len(locations) != 1:
            raise ValueError(f'Тара {bin_id} имеет разные этажи/улицы')
        bins[bin_id] = dict(location=locations.pop(),
            qty=float(rows['Количество'].sum()),
            load=float((rows['Количество'] / rows['Вместимость']).sum()),
            codes={normalize(v) for v in rows['Код']},
            groups={(normalize(s), normalize(g)) for s, g in zip(rows['Сектор'], rows['Группа'])},
            sectors={normalize(v) for v in rows['Сектор']},
            sources=[], target=None, reason=None)
    # На каждом шаге выбираем лучшую пару глобально. Даже более близкая
    # тара с общим сектором не обгоняет доступную пару с одинаковым кодом.
    while True:
        best = None
        for source_id, source in bins.items():
            if source['target'] is not None or source['sources'] or source['qty'] <= 0:
                continue
            if max_small_qty is not None and source['qty'] > max_small_qty:
                continue
            for target_id, target in bins.items():
                if target_id == source_id or target['target'] is not None:
                    continue
                zone, street = source['location']
                target_zone, target_street = target['location']
                distance = abs(street - target_street)
                if zone != target_zone or distance > max_street_distance:
                    continue
                if source['load'] + target['load'] > 1 + 1e-9:
                    continue
                if source['codes'] & target['codes']:
                    priority, reason = 3, 'Одинаковый код'
                elif source['groups'] & target['groups']:
                    priority, reason = 2, 'Общая группа в секторе'
                elif source['sectors'] & target['sectors']:
                    priority, reason = 1, 'Общий сектор'
                else:
                    continue
                score = (priority, -distance, target['load'] + source['load'],
                         -source['qty'], str(source_id), str(target_id))
                if best is None or score > best[0]:
                    best = score, source_id, target_id, reason
        if best is None:
            break
        _, source_id, target_id, reason = best
        source, target = bins[source_id], bins[target_id]
        source['target'], source['reason'] = target_id, reason
        target['sources'].append(source_id)
        target['load'] += source['load']
        target['qty'] += source['qty']
        for key in ['codes', 'groups', 'sectors']:
            target[key].update(source[key])
    groups = {}
    membership = {}
    for target_id, target in bins.items():
        if target['sources']:
            group_id = len(groups) + 1
            groups[group_id] = dict(target_bin=target_id, source_bins=target['sources'],
                                   fill_fraction=target['load'])
            for bin_id in [target_id] + target['sources']:
                membership[bin_id] = group_id
    result['Целевая_тара'] = [bins[b]['target'] if bins[b]['target'] is not None else b for b in result['Тара']]
    result['Группа_объединения'] = result['Тара'].map(membership).fillna(0).astype(int)
    result['Нужно_переместить'] = [bins[b]['target'] is not None for b in result['Тара']]
    result['Тип_перемещения'] = [bins[b]['reason'] or 'Не перемещается' for b in result['Тара']]
    result['Доля_объема'] = result['Количество'] / result['Вместимость']
    result['Заполнение_целевой_тары'] = result.groupby('Целевая_тара')['Доля_объема'].transform('sum')
    freed = {b for b, info in bins.items() if info['target'] is not None}
    stats = {'перемещено_тар': len(freed), 'создано_групп': len(groups),
             'тар_с_исходным_переполнением': sum(float((rows['Количество'] / rows['Вместимость']).sum()) > 1 + 1e-9 for _, rows in result.groupby('Тара'))}
    return result, freed, stats, groups


def create_consolidation_report(result, groups):
    rows = []
    for group_id, group in groups.items():
        target = group['target_bin']
        for bin_id in [target] + group['source_bins']:
            items = result[result['Тара'] == bin_id]
            rows.append({'Группа': group_id, 'Тара': bin_id,
                'Адрес': items['Адрес'].iloc[0], 'Целевая_тара': target,
                'Адрес_назначения': result.loc[result['Тара'] == target, 'Адрес'].iloc[0],
                'Количество': items['Количество'].sum(),
                'Действие': 'Оставить' if bin_id == target else 'Переместить целиком',
                'Причина': items['Тип_перемещения'].iloc[0],
                'Заполнение_после': group['fill_fraction']})
    return pd.DataFrame(rows, columns=['Группа', 'Тара', 'Адрес', 'Целевая_тара', 'Адрес_назначения', 'Количество', 'Действие', 'Причина', 'Заполнение_после'])


def write_warehouse_sheet(writer, result, groups):
    """Готовое задание складу: получатель («да») и целые тары-источники.

    Включены только группы с переносами. Итог учитывает исходный остаток
    получателя и все перемещаемые товары. Остальные тары на лист не попадают.
    """
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    workbook = writer.book
    ws = workbook.create_sheet('Для склада', 0)
    headers = ['признак объединения', 'Адрес', 'Тара', 'Сумма по количеству', 'куда']
    ws.append(headers)
    gray = PatternFill('solid', fgColor='BFBFBF')
    green = PatternFill('solid', fgColor='E2F0D9')
    thin = Side(style='thin', color='B7B7B7')
    thick = Side(style='medium', color='000000')
    for cell in ws[1]:
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = Border(left=thin, right=thin, top=thin, bottom=thick)
    ws.row_dimensions[1].height = 34
    for group_id, group in groups.items():
        sources = group['source_bins']
        if not sources:
            continue
        target = group['target_bin']
        bin_ids = [target] + list(sources)
        total = result.loc[result['Тара'].isin(bin_ids), 'Количество'].sum()
        ws.append([int(group_id), None, None, float(total), None])
        subtotal_row = ws.max_row
        for cell in ws[subtotal_row]:
            cell.fill = gray
            cell.font = Font(bold=True)
            cell.border = Border(top=thick, bottom=thin)
        for bin_id in bin_ids:
            items = result[result['Тара'] == bin_id]
            addresses = list(dict.fromkeys(items['Адрес'].astype(str)))
            identifier = bin_id.item() if hasattr(bin_id, 'item') else bin_id
            ws.append([int(group_id), '; '.join(addresses), identifier,
                       float(items['Количество'].sum()), 'да' if bin_id == target else None])
            for cell in ws[ws.max_row]:
                cell.border = Border(left=thin, right=thin)
                if bin_id == target:
                    cell.fill = green
        for cell in ws[ws.max_row]:
            cell.border = Border(left=thin, right=thin, bottom=thick)
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical='center', horizontal='left' if cell.column in (2, 5) else 'right')
        row[2].number_format = '0'
        row[3].number_format = '0.########'
    for column, width in {'A': 21, 'B': 25, 'C': 18, 'D': 23, 'E': 10}.items():
        ws.column_dimensions[column].width = width
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False
    ws.print_title_rows = '1:1'
    ws.print_options.horizontalCentered = True
    ws.sheet_properties.pageSetUpPr.fitToPage = True
    ws.page_setup.orientation = 'portrait'
    ws.page_setup.paperSize = ws.PAPERSIZE_A4
    ws.page_setup.fitToWidth = 1
    ws.page_setup.fitToHeight = 0
    ws.print_area = ws.dimensions
    workbook.active = 0
    return ws


def write_capacity_sheet(writer, original):
    """Справочник средних вместимостей из входного файла, не оценка габаритов."""
    data = original[['Сектор', 'Код', 'Вместимость']].copy()
    data['Сектор'] = data['Сектор'].astype(str).str.strip()
    data['Вместимость'] = pd.to_numeric(data['Вместимость'], errors='raise')
    # Каждый артикул имеет одинаковый вес независимо от числа строк остатков.
    by_code = data.groupby(['Сектор', 'Код'], as_index=False)['Вместимость'].mean()
    summary = by_code.groupby('Сектор')['Вместимость'].agg(
        **{'Средняя вместимость, ед.': 'mean', 'Минимум, ед.': 'min',
           'Максимум, ед.': 'max', 'Количество кодов': 'size'}).reset_index()
    summary['Источник'] = 'Столбец «Вместимость» исходного Excel'
    summary.to_excel(writer, sheet_name='Вместимость секторов', index=False)
    ws = writer.book['Вместимость секторов']
    from openpyxl.styles import Alignment, Font
    for c in ws[1]:
        c.font = Font(bold=True)
        c.alignment = Alignment(wrap_text=True)
    for col in ['B', 'C', 'D']:
        ws.column_dimensions[col].width = 24
        for cell in list(ws[col])[1:]:
            cell.number_format = '0.00'
    ws.column_dimensions['A'].width = 28
    ws.column_dimensions['E'].width = 22
    ws.column_dimensions['F'].width = 48
    ws.auto_filter.ref = ws.dimensions
    ws.freeze_panes = 'A2'
    return summary


In [ ]:
# Настройки: пути относительно папки блокнота
INPUT_FILE = Path('shlop.xlsx')
OUTPUT_FILE = Path('../outputs/boxes_match/схлопывание.xlsx')
MAX_SMALL_QTY = 6
MAX_STREET_DISTANCE = 1  # 0: только одна улица; 1: также соседние
STREET_COMPONENT = 'after_dash'

original = pd.read_excel(INPUT_FILE)
# Тары с отрицательными остатками требуют сверки, целиком исключаем их
# из автоматических переносов и сохраняем отдельно, ничего не обнуляя.
bad_bins = set(original.loc[pd.to_numeric(original['Количество'], errors='raise') < 0, 'Тара'])
invalid_rows = original[original['Тара'].isin(bad_bins)].copy()
valid_rows = original[~original['Тара'].isin(bad_bins)].copy()
print(f'Тар с отрицательными остатками (нужна сверка): {len(bad_bins)}')
df_result, freed_bins, moves_stats, consolidation_groups = optimize_whole_bins(
    valid_rows, max_small_qty=MAX_SMALL_QTY,
    max_street_distance=MAX_STREET_DISTANCE, street_component=STREET_COMPONENT)
report = create_consolidation_report(df_result, consolidation_groups)
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
    write_warehouse_sheet(writer, df_result, consolidation_groups)
    write_capacity_sheet(writer, original)
    df_result.to_excel(writer, sheet_name='Товары', index=False)
    report.to_excel(writer, sheet_name='План перемещений', index=False)
    invalid_rows.to_excel(writer, sheet_name='Требуют сверки', index=False)
    pd.DataFrame([moves_stats]).to_excel(writer, sheet_name='Статистика', index=False)
print(moves_stats)
if moves_stats['тар_с_исходным_переполнением']:
    print('Внимание: исходные тары уже превышают расчётную вместимость. Они не получают новые товары.')
print(f'Отчёт: {OUTPUT_FILE.resolve()}')
display(report.head(20))
